In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Load datasets
features = pd.read_parquet("data/processed/features.parquet")
folds = pd.read_parquet("data/processed/folds.parquet")

print(features.head())
print(features.columns)

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/features.parquet'

In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith((".parquet", ".csv", ".pkl")):
            print(os.path.join(root, file))

/kaggle/input/datasets/thaidatta/member-2/data/processed_local/features.parquet
/kaggle/input/datasets/thaidatta/member-2/data/processed_local/folds.parquet
/kaggle/input/datasets/thaidatta/member-2/data/processed_local/point_labels.parquet


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if "features.parquet" in files:
        feature_path = os.path.join(root, "features.parquet")
        print("Found:", feature_path)

Found: /kaggle/input/datasets/thaidatta/member-2/data/processed_local/features.parquet


In [4]:
import pandas as pd

df = pd.read_parquet(feature_path)

print(df.shape)
print(df.head())
print(df.columns)

(2881651, 16)
   arc_id   distance  rank                           trip_id  obs_idx  \
0   17630   6.114291     1  student_test__00000000__00000000        0   
1   17634   6.114291     2  student_test__00000000__00000000        0   
2   17633  46.000553     5  student_test__00000000__00000000        0   
3     212  46.000553     6  student_test__00000000__00000000        0   
4     211  79.017191     7  student_test__00000000__00000000        0   

   frac_along  arc_length  dist_rel_min  n_cand  heading_diff  \
0    0.872796  358.420308  0.000000e+00       8     14.067254   
1    0.127204  358.420308  3.552714e-15       8    165.932746   
2    0.000000   33.193140  3.988626e+01       8     14.183294   
3    1.000000   33.193140  3.988626e+01       8    165.816706   
4    0.000000   69.764589  7.290290e+01       8     25.214974   

   heading_diff_undirected  heading_valid      speed  d_prev     d_next  label  
0                14.067254              1  23.644427     0.0  23.644427    

In [8]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "/kaggle/input/datasets/thaidatta/member-2/data/processed_local/features.parquet"
)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_columns = [
    "distance",
    "rank",
    "frac_along",
    "arc_length",
    "dist_rel_min",
    "n_cand",
    "heading_diff",
    "heading_diff_undirected",
    "heading_valid",
    "speed",
    "d_prev",
    "d_next"
]

# Take a sample for faster training
sample_df = df.sample(n=min(50000, len(df)), random_state=42)

X = sample_df[feature_columns]
y = sample_df["label"].astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Standardize features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train_scaled.shape)
print("Testing samples:", X_test_scaled.shape)
print("Label distribution:")
print(y_train.value_counts())

Training samples: (40000, 12)
Testing samples: (10000, 12)
Label distribution:
label
0    37216
1     2784
Name: count, dtype: int64


In [10]:
!pip install pymc -q

In [11]:
import pymc as pm
import numpy as np

n_features = X_train_scaled.shape[1]

with pm.Model() as bayesian_model:

    # Priors
    intercept = pm.Normal("intercept", mu=0, sigma=2)

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=n_features
    )

    # Model
    logits = intercept + pm.math.dot(X_train_scaled, beta)

    probability = pm.math.sigmoid(logits)

    # Likelihood
    likelihood = pm.Bernoulli(
        "likelihood",
        p=probability,
        observed=y_train.values
    )

    # Fast Bayesian inference
    approx = pm.fit(
        10000,
        method="advi",
        progressbar=True
    )

    posterior = approx.sample(1000)

print("Bayesian model trained successfully!")

Output()

Finished [100%]: Average Loss = 6,973.6


Bayesian model trained successfully!


In [12]:
from scipy.special import expit

beta_samples = posterior.posterior["beta"].values.reshape(
    -1, n_features
)

intercept_samples = posterior.posterior["intercept"].values.reshape(-1)

# Calculate predictions
logits = (
    intercept_samples[:, None]
    + beta_samples @ X_test_scaled.T
)

probability_samples = expit(logits)

# Average posterior probability
pred_prob = probability_samples.mean(axis=0)

# Binary prediction
predictions = (pred_prob >= 0.5).astype(int)

print("Predictions generated!")

Predictions generated!


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))
print("F1 Score:", f1_score(y_test, predictions))
print("ROC-AUC:", roc_auc_score(y_test, pred_prob))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

Accuracy: 0.9494
Precision: 0.9094827586206896
Recall: 0.3031609195402299
F1 Score: 0.4547413793103448
ROC-AUC: 0.9600751067394075

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      9304
           1       0.91      0.30      0.45       696

    accuracy                           0.95     10000
   macro avg       0.93      0.65      0.71     10000
weighted avg       0.95      0.95      0.94     10000



In [14]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": beta_samples.mean(axis=0)
})

feature_importance = feature_importance.sort_values(
    "Coefficient",
    ascending=False
)

display(feature_importance)

,Feature,Coefficient
3,arc_length,0.184099
9,speed,0.038353
8,heading_valid,0.028774
2,frac_along,0.015341
11,d_next,-0.046532
10,d_prev,-0.046640
7,heading_diff_undirected,-0.094360
5,n_cand,-0.141606
4,dist_rel_min,-0.352328
1,rank,-0.471372


In [15]:
# Get test data corresponding to predictions
test_results = sample_df.loc[X_test.index].copy()

test_results["pred_probability"] = pred_prob
test_results["predicted_label"] = predictions

# Select highest-scoring candidate for each GPS point
top_candidate = (
    test_results
    .sort_values("pred_probability", ascending=False)
    .groupby(["trip_id", "obs_idx"])
    .first()
    .reset_index()
)

# Top-1 candidate accuracy
top1_accuracy = top_candidate["label"].mean()

print("Top-1 Candidate Accuracy:", top1_accuracy)
print("Number of GPS points:", len(top_candidate))

Top-1 Candidate Accuracy: 0.07120870100554073
Number of GPS points: 9746
